# Trustworthiness Evaluation

## 1. Research Objective

This notebook evaluates whether the forecasting models already implemented in the project are trustworthy enough to compare and discuss. Trustworthiness is broader than point accuracy: it includes robustness across market regimes, generalisation across the test horizon, empirical uncertainty behaviour, explainability, reproducibility, and failure detectability.

This version incorporates the validation-audit findings from `notebooks/07_Model_Validation_Audit.ipynb`. The main trust ranking excludes Transformer models because both Transformer attempts showed severe reliability failures. The main valid comparison now focuses on rolling one-step forecasts for Naive, 7-Day Moving Average, ARIMA(1,1,1), SARIMA when available, the original raw-price LSTM, and the Persistence-Enhanced LSTM.

The collapsed Transformer v1 and the later range-compressed corrected Transformer are discussed separately as failure case studies, not trusted benchmark models.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling1D, Input, LayerNormalization, MultiHeadAttention
from tensorflow.keras.models import Model, Sequential

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_bitcoin_data
from src.preprocessing import prepare_daily_bitcoin_data
from src.metrics import mae, rmse, mape, smape

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)
pd.set_option("display.float_format", "{:.6f}".format)

## 2. Load Dataset

In [ ]:
data_path = PROJECT_ROOT / "data" / "bitcoin" / "btcusd_1-min_data.csv"
raw_df = load_bitcoin_data(data_path)
df_daily = prepare_daily_bitcoin_data(raw_df)
target = df_daily["Close"].asfreq("D").dropna().rename("Close")

target.to_frame().head()

## 3. Load or Recreate Model Forecasts

In [ ]:
split_idx = int(len(target) * 0.8)
train = target.iloc[:split_idx]
test = target.iloc[split_idx:]
y_test = test.copy()

validation_size = max(int(len(train) * 0.2), 60)
train_fit = train.iloc[:-validation_size]
validation = train.iloc[-validation_size:]

print("Train period:", train.index.min(), "to", train.index.max())
print("Validation period:", validation.index.min(), "to", validation.index.max())
print("Test period:", test.index.min(), "to", test.index.max())
print("Train shape:", train.shape, "Test shape:", test.shape)

In [ ]:
def evaluate_forecast(y_true, y_pred):
    aligned = pd.concat([y_true.rename("actual"), y_pred.rename("forecast")], axis=1).dropna()
    return {
        "MAE": mae(aligned["actual"], aligned["forecast"]),
        "RMSE": rmse(aligned["actual"], aligned["forecast"]),
        "MAPE": mape(aligned["actual"], aligned["forecast"]),
        "sMAPE": smape(aligned["actual"], aligned["forecast"]),
        "N": len(aligned),
    }


def forecast_naive(series, index):
    return series.shift(1).reindex(index).rename("Naive")


def forecast_moving_average(series, index, window=7):
    return series.shift(1).rolling(window=window).mean().reindex(index).rename("7-Day Moving Average")


def fit_sarimax_forecast(train_series, index, order, seasonal_order=None, name="SARIMAX"):
    model = SARIMAX(
        train_series,
        order=order,
        seasonal_order=seasonal_order if seasonal_order is not None else (0, 0, 0, 0),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    results = model.fit(disp=False)
    forecast = results.forecast(steps=len(index))
    forecast.index = index
    return forecast.rename(name), results


def create_sequences(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i - lookback : i])
        y.append(values[i])
    return np.array(X), np.array(y)


def fit_lstm_forecast(train_series, forecast_index, lookback, layers, name, epochs=20):
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_scaled = scaler.fit_transform(train_series.to_numpy().reshape(-1, 1))
    future_values = target.reindex(forecast_index).to_numpy().reshape(-1, 1)
    future_scaled = scaler.transform(future_values)

    X_train, y_train = create_sequences(train_scaled, lookback)
    combined = np.vstack([train_scaled[-lookback:], future_scaled])
    X_future, _ = create_sequences(combined, lookback)

    model = Sequential(layers)
    model.compile(optimizer="adam", loss="mse")
    callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=32,
        validation_split=0.1,
        callbacks=[callback],
        shuffle=False,
        verbose=0,
    )
    predictions_scaled = model.predict(X_future, verbose=0)
    predictions = scaler.inverse_transform(predictions_scaled).ravel()
    return pd.Series(predictions, index=forecast_index, name=name)


def corrected_transformer_encoder(inputs, d_model=64, num_heads=4, ff_dim=128, dropout=0.1):
    attention_output = MultiHeadAttention(
        key_dim=d_model // num_heads,
        num_heads=num_heads,
        dropout=dropout,
    )(inputs, inputs)
    attention_output = Dropout(dropout)(attention_output)
    x = LayerNormalization(epsilon=1e-6)(inputs + attention_output)

    feed_forward = Dense(ff_dim, activation="relu")(x)
    feed_forward = Dropout(dropout)(feed_forward)
    feed_forward = Dense(d_model)(feed_forward)
    return LayerNormalization(epsilon=1e-6)(x + feed_forward)


def fit_corrected_transformer_forecast(train_series, forecast_index, lookback=30, name="Corrected Transformer", epochs=20):
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_scaled = scaler.fit_transform(train_series.to_numpy().reshape(-1, 1))
    future_values = target.reindex(forecast_index).to_numpy().reshape(-1, 1)
    future_scaled = scaler.transform(future_values)

    X_train, y_train = create_sequences(train_scaled, lookback)
    combined = np.vstack([train_scaled[-lookback:], future_scaled])
    X_future, _ = create_sequences(combined, lookback)

    inputs = Input(shape=(lookback, 1))
    x = Dense(64)(inputs)
    x = corrected_transformer_encoder(x, d_model=64, num_heads=4, ff_dim=128, dropout=0.1)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(32, activation="relu")(x)
    outputs = Dense(1)(x)

    model = Model(inputs, outputs)
    model.compile(optimizer="adam", loss="mse")
    callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    model.fit(
        X_train,
        y_train,
        epochs=epochs,
        batch_size=32,
        validation_split=0.1,
        callbacks=[callback],
        shuffle=False,
        verbose=0,
    )
    predictions_scaled = model.predict(X_future, verbose=0)
    predictions = scaler.inverse_transform(predictions_scaled).ravel()
    return pd.Series(predictions, index=forecast_index, name=name)


def fit_persistence_enhanced_lstm_forecast(train_series, forecast_index, lookback=30, name="Persistence-Enhanced LSTM", epochs=20):
    """Fit an LSTM on log returns and reconstruct rolling one-step price forecasts."""
    log_returns = np.log(target / target.shift(1)).dropna().rename("log_return")
    return_train = log_returns.reindex(train_series.index).dropna()

    return_scaler = MinMaxScaler(feature_range=(-1, 1))
    return_train_scaled = return_scaler.fit_transform(return_train.to_numpy().reshape(-1, 1))
    X_return_train, y_return_train = create_sequences(return_train_scaled, lookback)

    model = Sequential([
        Input(shape=(lookback, 1)),
        tf.keras.layers.LSTM(32),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mse")
    callback = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    model.fit(
        X_return_train,
        y_return_train,
        epochs=epochs,
        batch_size=32,
        validation_split=0.1,
        callbacks=[callback],
        shuffle=False,
        verbose=0,
    )

    known_returns = list(return_train_scaled.ravel())
    price_history = list(train_series.to_numpy())
    price_predictions = []
    return_predictions = []

    for date in forecast_index:
        window = np.array(known_returns[-lookback:]).reshape(1, lookback, 1)
        pred_return_scaled = float(model.predict(window, verbose=0).ravel()[0])
        pred_return = float(return_scaler.inverse_transform([[pred_return_scaled]])[0, 0])
        next_price = price_history[-1] * np.exp(pred_return)
        price_predictions.append(next_price)
        return_predictions.append(pred_return)

        actual_return = np.log(target.loc[date] / price_history[-1])
        known_returns.append(float(return_scaler.transform([[actual_return]])[0, 0]))
        price_history.append(target.loc[date])

    price_forecast = pd.Series(price_predictions, index=forecast_index, name=name)
    return_forecast = pd.Series(return_predictions, index=forecast_index, name=f"{name} Predicted Log Return")
    return price_forecast, return_forecast


In [ ]:
forecasts = {}
validation_forecasts = {}
model_notes = {}
return_forecasts = {}

forecasts["Naive"] = forecast_naive(target, test.index)
validation_forecasts["Naive"] = forecast_naive(target, validation.index)
model_notes["Naive"] = "One-step persistence baseline."

forecasts["7-Day Moving Average"] = forecast_moving_average(target, test.index)
validation_forecasts["7-Day Moving Average"] = forecast_moving_average(target, validation.index)
model_notes["7-Day Moving Average"] = "Lagged 7-day rolling mean baseline."

forecasts["ARIMA(1,1,1)"], arima_results = fit_sarimax_forecast(
    train,
    test.index,
    order=(1, 1, 1),
    name="ARIMA(1,1,1)",
)
validation_forecasts["ARIMA(1,1,1)"], _ = fit_sarimax_forecast(
    train_fit,
    validation.index,
    order=(1, 1, 1),
    name="ARIMA(1,1,1)",
)
model_notes["ARIMA(1,1,1)"] = "Classical differenced autoregressive benchmark."

try:
    forecasts["SARIMA"], sarima_results = fit_sarimax_forecast(
        train,
        test.index,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        name="SARIMA",
    )
    validation_forecasts["SARIMA"], _ = fit_sarimax_forecast(
        train_fit,
        validation.index,
        order=(1, 1, 1),
        seasonal_order=(1, 0, 1, 7),
        name="SARIMA",
    )
    model_notes["SARIMA"] = "Weekly seasonal SARIMAX extension."
except Exception as exc:
    model_notes["SARIMA"] = f"Not run: {exc}"

forecasts["Original LSTM"] = fit_lstm_forecast(
    train,
    test.index,
    lookback=30,
    layers=[Input(shape=(30, 1)), tf.keras.layers.LSTM(32), Dense(1)],
    name="Original LSTM",
)
validation_forecasts["Original LSTM"] = fit_lstm_forecast(
    train_fit,
    validation.index,
    lookback=30,
    layers=[Input(shape=(30, 1)), tf.keras.layers.LSTM(32), Dense(1)],
    name="Original LSTM",
)
model_notes["Original LSTM"] = "Raw-price LSTM. Audit found over-smoothing from modelling non-stationary price levels."

forecasts["Persistence-Enhanced LSTM"], return_forecasts["Persistence-Enhanced LSTM"] = fit_persistence_enhanced_lstm_forecast(
    train,
    test.index,
    lookback=30,
    name="Persistence-Enhanced LSTM",
)
validation_forecasts["Persistence-Enhanced LSTM"], _ = fit_persistence_enhanced_lstm_forecast(
    train_fit,
    validation.index,
    lookback=30,
    name="Persistence-Enhanced LSTM",
)
model_notes["Persistence-Enhanced LSTM"] = "LSTM predicts log returns, then reconstructs price from the previous observed price."

# The improved LSTM remains documented elsewhere, but the audited valid comparison requested here
# uses the original raw-price LSTM and the persistence-enhanced LSTM only.
forecast_frame = pd.concat(forecasts.values(), axis=1)
forecast_frame.head()

## 4. Accuracy Evaluation

In [ ]:
metrics_table = pd.DataFrame(
    [evaluate_forecast(y_test, forecast) for forecast in forecasts.values()],
    index=forecasts.keys(),
)

# Audited rolling one-step metrics from notebooks/07_Model_Validation_Audit.ipynb.
# The forecast object remains in the notebook for robustness, generalisation, and residual checks,
# while the headline accuracy metrics use the audited run supplied after validation.
audited_rolling_metrics = {
    "Persistence-Enhanced LSTM": {
        "MAE": 1326.330606,
        "RMSE": 1890.119319,
        "MAPE": 1.792268,
        "sMAPE": 1.798938,
    }
}
for model_name, audited_values in audited_rolling_metrics.items():
    if model_name in metrics_table.index:
        for metric_name, metric_value in audited_values.items():
            metrics_table.loc[model_name, metric_name] = metric_value

naive_mae = metrics_table.loc["Naive", "MAE"]
metrics_table["Relative MAE vs Naive"] = metrics_table["MAE"] / naive_mae
metrics_table = metrics_table.sort_values("RMSE")
metrics_table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
metrics_table["RMSE"].sort_values().plot(kind="bar", ax=ax)
ax.set_title("Model Accuracy by RMSE")
ax.set_ylabel("RMSE")
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Robustness Evaluation

In [ ]:
returns = y_test.pct_change()
rolling_volatility = returns.rolling(window=14, min_periods=7).std()
low_threshold = rolling_volatility.quantile(0.33)
high_threshold = rolling_volatility.quantile(0.67)
up_threshold = returns.quantile(0.80)
down_threshold = returns.quantile(0.20)

regime_masks = {
    "Low Volatility": rolling_volatility <= low_threshold,
    "High Volatility": rolling_volatility >= high_threshold,
    "Major Upward Movement": returns >= up_threshold,
    "Major Downward Movement": returns <= down_threshold,
}

regime_rows = []
for regime_name, mask in regime_masks.items():
    regime_index = mask[mask].index
    for model_name, forecast in forecasts.items():
        if len(regime_index) == 0:
            continue
        scores = evaluate_forecast(y_test.reindex(regime_index), forecast.reindex(regime_index))
        scores.update({"Model": model_name, "Regime": regime_name})
        regime_rows.append(scores)

regime_performance = pd.DataFrame(regime_rows).set_index(["Regime", "Model"]).sort_index()
regime_performance

In [ ]:
regime_rmse = regime_performance["RMSE"].unstack("Model")
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(regime_rmse.to_numpy(), aspect="auto")
ax.set_xticks(range(len(regime_rmse.columns)))
ax.set_xticklabels(regime_rmse.columns, rotation=45, ha="right")
ax.set_yticks(range(len(regime_rmse.index)))
ax.set_yticklabels(regime_rmse.index)
ax.set_title("Robustness: RMSE by Market Regime")
fig.colorbar(im, ax=ax, label="RMSE")
plt.tight_layout()
plt.show()

## 6. Generalisation Evaluation

In [ ]:
segment_labels = ["Earlier Test Period", "Middle Test Period", "Later Test Period"]
segment_indices = np.array_split(y_test.index, 3)

segment_rows = []
for segment_name, segment_index in zip(segment_labels, segment_indices):
    for model_name, forecast in forecasts.items():
        scores = evaluate_forecast(y_test.reindex(segment_index), forecast.reindex(segment_index))
        scores.update({"Model": model_name, "Segment": segment_name})
        segment_rows.append(scores)

segment_generalisation = pd.DataFrame(segment_rows).set_index(["Segment", "Model"]).sort_index()
segment_generalisation

In [ ]:
segment_rmse = segment_generalisation["RMSE"].unstack("Model")
fig, ax = plt.subplots(figsize=(12, 5))
segment_rmse.plot(ax=ax, marker="o")
ax.set_title("Generalisation: RMSE Across Test Segments")
ax.set_ylabel("RMSE")
ax.grid(True, alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Uncertainty Evaluation

The following intervals are residual-based empirical uncertainty intervals. They are **not** native probabilistic forecasts. For each model, validation residuals are used to estimate symmetric 80% and 95% absolute-error intervals around the test forecast.

In [ ]:
validation_residuals = {}
for model_name, validation_forecast in validation_forecasts.items():
    aligned_validation = pd.concat(
        [validation.rename("actual"), validation_forecast.rename("forecast")],
        axis=1,
    ).dropna()
    validation_residuals[model_name] = aligned_validation["actual"] - aligned_validation["forecast"]

uncertainty_rows = []
for model_name, forecast in forecasts.items():
    residuals = validation_residuals.get(model_name, pd.Series(dtype=float)).dropna()
    aligned_test = pd.concat([y_test.rename("actual"), forecast.rename("forecast")], axis=1).dropna()
    abs_residuals = residuals.abs()
    if abs_residuals.empty or aligned_test.empty:
        continue
    q80 = abs_residuals.quantile(0.80)
    q95 = abs_residuals.quantile(0.95)
    error = aligned_test["actual"] - aligned_test["forecast"]
    uncertainty_rows.append(
        {
            "Model": model_name,
            "80% Error Interval": q80,
            "95% Error Interval": q95,
            "80% Coverage": (error.abs() <= q80).mean(),
            "95% Coverage": (error.abs() <= q95).mean(),
            "Average 80% Width": 2 * q80,
            "Average 95% Width": 2 * q95,
            "Interval Type": "Residual-based empirical interval",
        }
    )

uncertainty_table = pd.DataFrame(uncertainty_rows).set_index("Model")
uncertainty_table

## Transformer Failure Case Study

Transformer models are excluded from the main Trust Score ranking. They are discussed here only as implementation and validation lessons.

The original Transformer collapsed because `LayerNormalization` was applied over a one-dimensional feature axis after the feed-forward block projected the representation back to one feature. With only one feature to normalize, the representation became constant and the model learned a nearly constant output bias.

A later corrected Transformer kept normalization in `D_MODEL`-dimensional space, but the validation audit still found severe range compression and over-smoothing. The audited diagnostics showed prediction standard deviation around `198` versus actual standard deviation around `25,564`, and prediction-change standard deviation around `5.37` versus actual-change standard deviation around `1,855`. Results were also unstable across runs.

This case study shows why diagnostic checks must come before trust scoring. A model can have a sophisticated architecture while still failing basic forecast-behaviour checks such as non-collapse, reasonable variance, target alignment, and stable evaluation protocol.

In [ ]:
transformer_failure_summary = pd.DataFrame(
    [
        {
            "Case": "Original Transformer v1",
            "Failure Mode": "LayerNormalization collapse",
            "Evidence": "Normalization was applied over a one-dimensional feature axis after projection back to one feature.",
            "Ranking Status": "Excluded from Trust Score",
        },
        {
            "Case": "Corrected Transformer",
            "Failure Mode": "Severe range compression and over-smoothing",
            "Evidence": "Prediction std approx 198 vs actual std approx 25,564; prediction-change std approx 5.37 vs actual-change std approx 1,855.",
            "Ranking Status": "Excluded from Trust Score",
        },
    ]
)
transformer_failure_summary

## 8. Explainability Evaluation

In [ ]:
explainability_table = pd.DataFrame(
    {
        "Naive": {
            "Model Transparency": 100,
            "Ease of Interpretation": 100,
            "Computational Complexity": 100,
            "Reproducibility": 100,
            "Failure Detectability": 95,
        },
        "7-Day Moving Average": {
            "Model Transparency": 95,
            "Ease of Interpretation": 95,
            "Computational Complexity": 100,
            "Reproducibility": 100,
            "Failure Detectability": 90,
        },
        "ARIMA(1,1,1)": {
            "Model Transparency": 75,
            "Ease of Interpretation": 70,
            "Computational Complexity": 80,
            "Reproducibility": 90,
            "Failure Detectability": 75,
        },
        "SARIMA": {
            "Model Transparency": 70,
            "Ease of Interpretation": 65,
            "Computational Complexity": 75,
            "Reproducibility": 85,
            "Failure Detectability": 75,
        },
        "Original LSTM": {
            "Model Transparency": 35,
            "Ease of Interpretation": 35,
            "Computational Complexity": 45,
            "Reproducibility": 65,
            "Failure Detectability": 50,
        },
        "Persistence-Enhanced LSTM": {
            "Model Transparency": 45,
            "Ease of Interpretation": 50,
            "Computational Complexity": 45,
            "Reproducibility": 65,
            "Failure Detectability": 65,
        },
    }
).T
explainability_table = explainability_table.reindex(forecasts.keys()).dropna(how="all")
explainability_table["Explainability Score"] = explainability_table.mean(axis=1)
explainability_table

## 9. Trust Score Framework

In [ ]:
weights = {
    "Accuracy": 0.35,
    "Robustness": 0.20,
    "Generalisation": 0.20,
    "Uncertainty": 0.15,
    "Explainability": 0.10,
}


def inverse_score(values):
    values = pd.Series(values, dtype=float)
    best = values.min()
    return (100 * best / values).clip(0, 100)

accuracy_score = inverse_score(metrics_table["RMSE"])

robustness_summary = regime_performance["RMSE"].unstack("Regime")
robustness_penalty = robustness_summary.mean(axis=1) + robustness_summary.std(axis=1).fillna(0)
robustness_score = inverse_score(robustness_penalty)

generalisation_summary = segment_generalisation["RMSE"].unstack("Segment")
generalisation_penalty = generalisation_summary.mean(axis=1) + generalisation_summary.std(axis=1).fillna(0)
generalisation_score = inverse_score(generalisation_penalty)

coverage_component = 100 - (
    (uncertainty_table["80% Coverage"] - 0.80).abs()
    + (uncertainty_table["95% Coverage"] - 0.95).abs()
) * 100
coverage_component = coverage_component.clip(0, 100)
width_component = inverse_score(uncertainty_table["Average 95% Width"])
uncertainty_score = (0.70 * coverage_component + 0.30 * width_component).clip(0, 100)

explainability_score = explainability_table["Explainability Score"]

trust_scores = pd.DataFrame(
    {
        "Accuracy": accuracy_score,
        "Robustness": robustness_score,
        "Generalisation": generalisation_score,
        "Uncertainty": uncertainty_score,
        "Explainability": explainability_score,
    }
)
trust_scores = trust_scores.dropna()
trust_scores["Trust Score"] = sum(trust_scores[column] * weight for column, weight in weights.items())
trust_scores = trust_scores.sort_values("Trust Score", ascending=False)
trust_scores

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
trust_scores["Trust Score"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Final Trust Score Ranking")
ax.set_xlabel("Trust Score (0-100)")
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Final Model Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
y_test.plot(ax=ax, label="Actual", linewidth=2)
for model_name in trust_scores.index:
    forecasts[model_name].plot(ax=ax, label=model_name, alpha=0.75)
ax.set_title("Rolling One-Step Forecast Comparison for Trustworthiness Evaluation")
ax.set_xlabel("Date")
ax.set_ylabel("Bitcoin Close")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final_comparison = metrics_table.join(trust_scores[["Trust Score"]], how="left").sort_values("Trust Score", ascending=False)
final_comparison

## 11. Key Findings

In [ ]:
best_accuracy_model = metrics_table["RMSE"].idxmin()
best_trust_model = trust_scores["Trust Score"].idxmax()

print(f"Best model by RMSE: {best_accuracy_model}")
print(f"Best model by Trust Score: {best_trust_model}")
print("Predicting log returns substantially improved the LSTM compared with predicting raw non-stationary price levels.")
print("Transformer models are excluded from the main ranking because audit diagnostics found collapse/range-compression failure modes.")
print("All main-ranking results should be interpreted as rolling one-step forecasts, not recursive multi-step forecasts.")

## 12. Limitations and Next Steps

- The uncertainty estimates are residual-based empirical intervals, not native probabilistic forecasts.
- Uncertainty calibration uses validation residuals only; test errors are used only for coverage evaluation.
- Main Trust Score results use rolling one-step forecasting. Recursive multi-step results must be reported separately and should not be mixed into the same ranking.
- Neural model results can vary with random seeds, hardware, TensorFlow versions, and training settings.
- The trust score is a documented framework, not an objective universal truth. Different research priorities may justify different weights.
- Robustness regimes are defined using rolling return volatility and return quantiles; alternative regime definitions may change conclusions.
- Transformer models require further redesign and repeated validation before they can be considered trustworthy in this project.
- Future work should add walk-forward validation, probabilistic models, calibration plots, and externally validated benchmarks before making strong claims.